# Showjumping SSL — Milestone Notebook

End-to-end on Colab Pro (T4/L4 GPU). Clones the repo from GitHub for the **code**, mounts Drive for the **large data + checkpoints** (so videos don't have to re-download every session).

Edit the two variables in cell 0 (`GITHUB_REPO` and `DRIVE_DATA_ROOT`) before running.

## 0. Clone repo + mount Drive for data

Layout after this cell:
```
/content/project/                  <- cloned from GitHub (fresh each session)
├── src/, notebooks/, milestone/, requirements.txt
├── data/      -> symlink to /content/drive/MyDrive/CS131/data
└── checkpoints/ -> symlink to /content/drive/MyDrive/CS131/checkpoints
```

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_REPO = 'https://github.com/bballhaus/showjumping-ssl.git'
DRIVE_DATA_ROOT = '/content/drive/MyDrive/CS131'   # data + checkpoints live here
BRANCH = 'main'

# 0a. Mount Drive (data + checkpoints persist here).
from google.colab import drive
drive.mount('/content/drive')
Path(f'{DRIVE_DATA_ROOT}/data').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_DATA_ROOT}/checkpoints').mkdir(parents=True, exist_ok=True)

# 0b. Clone (or pull) the code repo to /content/project.
REPO_DIR = Path('/content/project')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1',
                    GITHUB_REPO, str(REPO_DIR)], check=True)

# 0c. Wire repo's data/ and checkpoints/ to Drive via symlinks.
for sub in ['data', 'checkpoints']:
    target = Path(f'{DRIVE_DATA_ROOT}/{sub}')
    link = REPO_DIR / sub
    if link.is_symlink() or link.exists():
        if link.is_dir() and not link.is_symlink():
            subprocess.run(['rm', '-rf', str(link)], check=True)
        else:
            link.unlink(missing_ok=True)
    link.symlink_to(target, target_is_directory=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())
print('data ->', os.readlink('data'))
print('checkpoints ->', os.readlink('checkpoints'))

## 1. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg
!nvidia-smi

## 2. Scrape + segment + filter clips

Curate `src/data/clip_sources.txt` with YouTube URLs (FEI World Cup, Longines GCT, CSI 3* highlight reels at 1.50 m) before running.

- yt-dlp prints its own progress bar for downloads.
- `segment` shows tqdm bars (one over videos, one nested over clips per video).
- `filter_clips` runs YOLO on each clip and moves non-jumping clips (crowd shots, course walks, interviews) into `data/clips_rejected/`.

In [ ]:
import subprocess, time
from pathlib import Path
from tqdm.auto import tqdm

from src.data.scrape import read_sources, download_one, DEFAULT_SOURCES
from src.data.segment import segment_video
from src.data.filter_clips import filter_clips

RAW = Path('data/raw')
CLIPS = Path('data/clips')
REJECTED = Path('data/clips_rejected')
CLIP_LEN = 2.0
STRIDE = 30.0          # 30 s stride keeps ~240 clips per 2 h video
MIN_HORSE_FRAC = 0.4   # YOLO must see a horse in >=40% of sampled frames

# 2a. Download. yt-dlp prints its own % bar (expect ~5-10 min per 2h video on Colab).
urls = read_sources(DEFAULT_SOURCES)
print(f'[scrape] {len(urls)} videos to download -> {RAW}')
for url in tqdm(urls, desc='downloads', unit='vid'):
    download_one(url, RAW)

# 2b. Segment. tqdm shows one bar over videos, one nested over clips per video.
vids = sorted(RAW.glob('*.mp4'))
print(f'[segment] cutting {len(vids)} videos at stride={STRIDE}s ...')
total = 0
t0 = time.time()
for v in tqdm(vids, desc='segment', unit='vid'):
    n = segment_video(v, CLIPS, clip_len=CLIP_LEN, stride=STRIDE)
    tqdm.write(f'  {v.name}: {n} clips')
    total += n
print(f'[segment] {total} raw clips in {time.time()-t0:.0f}s')

# 2c. YOLO horse-presence filter. Moves non-jumping clips to data/clips_rejected/.
raw_count = len(list(CLIPS.glob('*.mp4')))
print(f'[filter] running YOLO on {raw_count} clips ...')
kept, rejected = filter_clips(CLIPS, REJECTED,
                              weights='yolov8n.pt',
                              device='cuda',
                              stride=4,
                              min_frac=MIN_HORSE_FRAC,
                              conf=0.35)
print(f'[filter] kept {kept}, rejected {rejected}')
print(f'\nFinal usable clips: {kept} (in {CLIPS})')

## 3. Hand-annotate fence boxes — in Colab

Inline annotator: drag a rectangle around the fence on the displayed frame, then click **Vertical** (1 pole) or **Oxer** (2 poles). Skip clips that aren't a clear side-on view. ~25–30 boxes is plenty for the milestone.

Annotations save to `data/annotations/fences.csv` (which is on Drive, so they persist) and the script resumes from where you left off if you re-run.

In [ ]:
# One-time per session: install ipympl + enable widgets + switch to interactive matplotlib.
!pip install -q ipympl ipywidgets
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib widget

from src.preprocess.annotate_colab import ColabAnnotator
ann = ColabAnnotator(
    clips_dir='data/clips',
    out_csv='data/annotations/fences.csv',
    limit=30,           # annotate up to 30 in this session
)
ann.start()

## 4. YOLO horse detection + geometric d on all clips

In [ ]:
!python -m src.preprocess.run_pipeline \
    --clips data/clips \
    --fences data/annotations/fences.csv \
    --out data/annotations/auto.csv \
    --weights yolov8n.pt \
    --device cuda

## 5. SSL pretraining

Small R(2+1)D-18 with InfoNCE on two augmented views + 6-way temporal-order classification. With ~400 clips and 20 epochs this finishes in ~15–25 min on a T4. **Fallback:** pass `--kinetics-init` if nearest-neighbor retrieval looks bad (see cell 7).

In [ ]:
!python -m src.ssl.train \
    --clips data/clips \
    --out checkpoints \
    --epochs 20 \
    --batch-size 16 \
    --lr 1e-3 \
    --tau 0.1 \
    --lambda-order 0.3 \
    --workers 4 \
    --device cuda

## 6. Generate embeddings

In [ ]:
!python -m src.ssl.embed \
    --clips data/clips \
    --ckpt checkpoints/encoder.pt \
    --out data/embeddings.npz \
    --device cuda

## 7. Nearest-neighbor sanity check

If retrieved neighbors share approach phase / fence type, the SSL encoder learned something useful. If they look like crowd shots or random frames, switch to `--kinetics-init` in cell 5.

In [ ]:
import numpy as np, random
data = np.load('data/embeddings.npz', allow_pickle=True)
F = data['features']; paths = list(data['paths'])
F_n = F / (np.linalg.norm(F, axis=1, keepdims=True) + 1e-9)
sim = F_n @ F_n.T
np.fill_diagonal(sim, -1)
q = random.randrange(len(paths))
nn = np.argsort(-sim[q])[:5]
print('query:', paths[q])
for i in nn:
    print(f'  sim={sim[q,i]:+.3f}  {paths[i]}')

## 8. Build all milestone figures

In [ ]:
!python -m src.viz.make_figures \
    --ckpt checkpoints/encoder.pt \
    --log checkpoints/train_log.csv \
    --emb data/embeddings.npz \
    --csv data/annotations/auto.csv \
    --out milestone/figures

!python -m src.viz.detection_examples \
    --clips data/clips \
    --fences data/annotations/fences.csv \
    --out-dir milestone/figures/det \
    --weights yolov8n.pt \
    --device cuda \
    --max 4

## 9. Preview figures

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('milestone/figures/*.png')):
    print(p)
    display(Image(p))
for p in sorted(glob.glob('milestone/figures/det/*.jpg'))[:3]:
    print(p)
    display(Image(p))

## 10. Compile the milestone PDF

Drops `milestone.pdf` into the cloned repo's `milestone/` folder. Copy it to Drive (or just download from Colab) to submit.

In [ ]:
!apt-get -qq install -y texlive-latex-base texlive-latex-extra texlive-fonts-recommended
!cd milestone && pdflatex -interaction=nonstopmode milestone.tex && ls -la milestone.pdf

# Copy compiled PDF back to Drive for easy submission.
import shutil, os
drive_out = '/content/drive/MyDrive/CS131/milestone.pdf'
shutil.copy('milestone/milestone.pdf', drive_out)
print(f'wrote {drive_out}')